# NISAR GCOV Data Discovery & Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bullocke/nice-sar/blob/main/notebooks/00_data_discovery.ipynb)

This notebook discovers NISAR GCOV data near **Salt Lake City, Utah** and validates that
`nice-sar` I/O functions work against real NISAR HDF5 files.

**Run this in Google Colab or CryoCloud.**

### Phase 3 Tasks Covered
| Task | Description |
|------|-------------|
| TASK-019 | Document granule ID and S3 path |
| TASK-020 | Test `open_nisar()` with S3 path |
| TASK-021 | Test `get_frequencies()` and `get_polarizations()` |
| TASK-022 | Test `read_gcov_metadata()` |
| TASK-023 | Test `get_projection_info()` |
| TASK-024 | Test `read_gcov()` |

## 1. Setup

In [ ]:
# Pin fsspec/s3fs to avoid Colab conflicts, then force-reinstall only nice-sar
%pip install -q "fsspec<=2025.3.0" "s3fs<=2025.3.0"
%pip install -q --force-reinstall --no-deps "git+https://github.com/bullocke/nice-sar.git@main"

In [ ]:
import logging
import os

# Colab pre-configures the root logger, so basicConfig() is a no-op.
# Force the root logger to show INFO messages.
logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s", force=True)

import earthaccess
import h5py
import numpy as np

from nice_sar.auth.earthdata import login, get_s3_filesystem, get_https_filesystem
from nice_sar.io.hdf5 import open_nisar, get_frequencies, get_polarizations
from nice_sar.io.products import (
    read_gcov,
    read_gcov_metadata,
    get_projection_info,
)

### Authenticating with NASA Earthdata

`nice_sar.auth.earthdata.login()` wraps `earthaccess.login()`, which checks for credentials in this order:

**Option 1 — Environment variables** (best for Colab / CI):
```bash
export EARTHDATA_USERNAME="your_username"
export EARTHDATA_PASSWORD="your_password"
```
In Colab, set these in a cell before calling `login()`:
```python
import os
os.environ["EARTHDATA_USERNAME"] = "your_username"
os.environ["EARTHDATA_PASSWORD"] = "your_password"
```

**Option 2 — `.netrc` file** (best for local / CHPC):
Add to `~/.netrc`:
```
machine urs.earthdata.nasa.gov
    login your_username
    password your_password
```
Then `chmod 600 ~/.netrc`.

**Option 3 — Interactive prompt** (fallback):
If no credentials are found, `earthaccess` will prompt you in the notebook.

> **Don't have an account?** Register free at https://urs.earthdata.nasa.gov/users/new

In [ ]:
# Authenticate with NASA Earthdata
# Uses env vars, .netrc, or interactive prompt (see above)
login()

## 2. Search for GCOV Data

We search for NISAR L2 GCOV granules near Salt Lake City.  
If none are found, we fall back to a global search.

In [ ]:
# Salt Lake City area bounding box (west, south, east, north)
SLC_BBOX = (-112.1, 40.5, -111.7, 40.9)

# Search using earthaccess
# Correct short_name for NISAR GCOV: NISAR_L2_GCOV_BETA_V1
# See https://nisar-docs.asf.alaska.edu/access/earthaccess/
results = earthaccess.search_data(
    short_name="NISAR_L2_GCOV_BETA_V1",
    bounding_box=SLC_BBOX,
    count=10,
)

logging.info("Found %d GCOV granules near SLC", len(results))
for i, r in enumerate(results):
    print(f"  [{i}] {r['umm']['GranuleUR']}")

# Find NISAR Data: https://nisar-docs.asf.alaska.edu/earthdata-search/

In [ ]:
# Fallback: search globally if none found near SLC
if len(results) == 0:
    logging.info("No GCOV data near SLC. Searching globally...")
    results = earthaccess.search_data(
        short_name="NISAR_L2_GCOV_BETA_V1",
        count=10,
    )
    logging.info("Found %d GCOV granules globally", len(results))
    for i, r in enumerate(results):
        print(f"  [{i}] {r['umm']['GranuleUR']}")

# If still nothing, try alternate collection short names
if len(results) == 0:
    for name in ["NISAR_L2_GCOV", "NISAR_GCOV", "NISAR_L2_PR_GCOV_V1"]:
        results = earthaccess.search_data(short_name=name, count=5)
        if results:
            logging.info("Found %d granules with short_name=%s", len(results), name)
            break

assert len(results) > 0, (
    "No GCOV data found. NISAR GCOV may not be publicly available yet. "
    "Check https://search.earthdata.nasa.gov for current NISAR collections."
)

## 3. Access and Open HDF5

**S3 vs HTTPS access:**
- **S3 direct read** works only from AWS `us-west-2` (CryoCloud, EC2). Fastest option.
- **HTTPS streaming** works from anywhere (Colab, local, CHPC). Slightly slower.

The code below auto-detects the environment and picks the right access method.

In [ ]:
# TASK-019: Document granule ID and access path
granule = results[0]
granule_id = granule["umm"]["GranuleUR"]
logging.info("Selected granule: %s", granule_id)

# Detect environment: S3 access only works from AWS us-west-2
# CryoCloud sets AWS_DEFAULT_REGION; we also check for common cloud indicators
IN_AWS_US_WEST_2 = os.environ.get("AWS_DEFAULT_REGION", "") == "us-west-2"

if IN_AWS_US_WEST_2:
    # S3 direct read (fastest, CryoCloud / EC2 in us-west-2)
    data_links = earthaccess.results.DataGranule.data_links(granule, access="direct")
    data_path = next((l for l in data_links if l.endswith(".h5")), data_links[0])
    fs = get_s3_filesystem()
    logging.info("Using S3 direct read: %s", data_path)
else:
    # HTTPS streaming (works anywhere: Colab, local, CHPC)
    data_links = earthaccess.results.DataGranule.data_links(granule, access="external")
    data_path = next((l for l in data_links if l.endswith(".h5")), data_links[0])
    fs = get_https_filesystem()
    logging.info("Using HTTPS streaming: %s", data_path)

In [ ]:
# TASK-020: Test open_nisar() with authenticated filesystem
h5 = open_nisar(data_path, filesystem=fs)

logging.info("Opened HDF5 file: %s", data_path)
logging.info("Top-level groups: %s", list(h5.keys()))
print("\n✓ open_nisar works")

## 4. Validate I/O Functions

### 4a. `get_frequencies()` and `get_polarizations()`

In [ ]:
# TASK-021: Test get_frequencies
freqs = get_frequencies(h5)
logging.info("Frequencies: %s", freqs)
assert isinstance(freqs, list), f"Expected list, got {type(freqs)}"
assert len(freqs) >= 1, "Expected at least one frequency"
print("✓ get_frequencies works")

# TASK-021: Test get_polarizations
for freq in freqs:
    pols = get_polarizations(h5, frequency=freq)
    logging.info("Frequency %s polarizations: %s", freq, pols)
    assert isinstance(pols, list), f"Expected list, got {type(pols)}"
    assert len(pols) >= 1, f"Expected at least one polarization for freq {freq}"
print("✓ get_polarizations works")

### 4b. `read_gcov_metadata()`

In [ ]:
# TASK-022: Test read_gcov_metadata
metadata = read_gcov_metadata(h5)
print("GCOV Metadata:")
for k, v in metadata.items():
    print(f"  {k}: {v}")

assert "product_type" in metadata
assert "orbit" in metadata
print("\n✓ read_gcov_metadata works")

### 4c. `get_projection_info()`

In [ ]:
# TASK-023: Test get_projection_info
crs, transform, x_coords, y_coords = get_projection_info(h5, frequency=freqs[0])

print(f"CRS:       {crs}")
print(f"EPSG:      {crs.to_epsg()}")
print(f"Transform: {transform}")
print(f"X coords:  shape={x_coords.shape}, range=[{x_coords[0]:.1f}, {x_coords[-1]:.1f}]")
print(f"Y coords:  shape={y_coords.shape}, range=[{y_coords[0]:.1f}, {y_coords[-1]:.1f}]")
print("\n✓ get_projection_info works")

### 4d. `read_gcov()`

In [ ]:
# TASK-024: Test read_gcov
# Use the first available polarization
pols = get_polarizations(h5, frequency=freqs[0])

# Map GCOV pol names (e.g. HHHH -> HH) for the read_gcov interface
pol_short = pols[0][:2] if len(pols[0]) == 4 else pols[0]
logging.info("Reading polarization: %s (from %s)", pol_short, pols[0])

# Close h5 -- read_gcov manages its own handle
h5.close()

# read_gcov opens the file, reads data, and closes it
da_xr = read_gcov(
    data_path,
    frequency=freqs[0],
    polarization=pol_short,
    chunks={"y": 512, "x": 512},
    filesystem=fs,
)

print(f"DataArray shape:  {da_xr.shape}")
print(f"Dims:             {da_xr.dims}")
print(f"Coords:           {list(da_xr.coords)}")
print(f"CRS:              {da_xr.attrs.get('crs')}")
print(f"Polarization:     {da_xr.attrs.get('polarization')}")
print(f"Dask-backed:      {hasattr(da_xr.data, 'dask')}")

In [ ]:
# Quick stats on a center subset (corners are often outside the swath / all NaN)
cy, cx = da_xr.shape[0] // 2, da_xr.shape[1] // 2
subset = da_xr[cy : cy + 256, cx : cx + 256].values

print(f"Subset shape: {subset.shape}")
print(f"  min:  {np.nanmin(subset):.6f}")
print(f"  max:  {np.nanmax(subset):.6f}")
print(f"  mean: {np.nanmean(subset):.6f}")
print(f"  NaN%: {np.isnan(subset).mean() * 100:.1f}%")
print("\n✓ read_gcov works")

## 5. Explore HDF5 Structure

Walk the full HDF5 tree to document the actual NISAR GCOV layout.

In [ ]:
# Re-open to explore the tree structure
h5 = open_nisar(data_path, filesystem=fs)


def print_h5_tree(group, prefix="", max_depth=4, _depth=0):
    """Print the HDF5 group tree up to max_depth."""
    if _depth >= max_depth:
        return
    for key in sorted(group):
        item = group[key]
        if isinstance(item, h5py.Group):
            print(f"{prefix}{key}/")
            print_h5_tree(item, prefix + "  ", max_depth, _depth + 1)
        elif isinstance(item, h5py.Dataset):
            shape_str = str(item.shape) if item.shape else "scalar"
            print(f"{prefix}{key}  [{shape_str} {item.dtype}]")


print_h5_tree(h5)
h5.close()

## 6. Summary

Record the results below for Phase 4 bug fixes.  
Copy any errors into the **Bug Log** in the Priority 1 plan.

In [ ]:
# ---- FILL IN AFTER RUNNING ----
# Granule ID:      <paste here>
# S3 path:         <paste here>
# Frequencies:     <paste here>
# Polarizations:   <paste here>
# CRS / EPSG:      <paste here>
# Data shape:       <paste here>
#
# ---- ERRORS / BUGS ----
# If any function raised an error, paste the traceback here.
# These will be fixed in Phase 4.
#
# ---- NOTES ----
# Any observations about the HDF5 structure that differ from
# our assumptions (paths, dtypes, attribute names, etc.)
#
logging.info("Done! Copy results above into the Phase 3 logs.")